# dbt　スタイルのデータ変換パイプライン

## 目的
- dbtの「層構造(raw→stagint →intermediate → marts)」の考え方を学ぶ
- Python + SQL　で同じアーキテクチャを実装する
- データ品質テスト(assertions) を自動化する
- Day 15-16　で本物のdbtに移行するための基礎を作る

## dbt とは
dbt (data build tool)は「SQL だけでデータ変換パイプラインを構築する」ツール。
以下の3つが特徴:
1. 層構造: raw → staging → intermediate → marts とデータを段階的に加工
2. テスト: データ品質を自動チェック (NULL がないか、重複がないか等)
3. ドキュメント:テーブルの説明を自動生成

##　層構造の考え方

In [1]:
# ライブラリのインポート
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

conn = sqlite3.connect('nikkei225.db')

count = pd.read_sql("SELECT COUNT(*) as cnt FROM raw_nikkei225", conn)
print(f"✓ データベース接続完了: {count['cnt'][0]} 行")
print(f"  実行時刻: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ データベース接続完了: 1766 行
  実行時刻: 2026-04-08 21:26:29


## 1. Staging 層:生データの整理

###役割
- rawデータの列名を統一する (大文字→小文字)
- 型を明確にする
- 欠損値を処理する
- rawデータは一切変更しない (stagingはrawのコピー + 整理)

### dbt　の場合：
'''sql
-- models/staging/stg_nikkei225.sql
SELECT
    date,
    CAST(Close AS REAL) as close_price,
    CAST(Volume AS INTEGER) as volume
FROM {{ source('raw', 'raw_nikkei225')}}
WHERE Close IS NOT NULL
'''

### Python + SQL　の場合は今日の実装

In [2]:
# Staging 層
print("[STAGING]生データの整理")

conn.execute("DROP TABLE IF EXISTS stg_nikkei225")

conn.execute("""
             CREATE TABLE stg_nikkei225 AS
    SELECT 
        date,
        ROUND(CAST(Open AS REAL), 2) as open_price,
        ROUND(CAST(High AS REAL), 2) as high_price,
        ROUND(CAST(Low AS REAL), 2) as low_price,
        ROUND(CAST(Close AS REAL), 2) as close_price,
        CAST(Volume AS INTEGER) as volume,
        SUBSTR(date, 1, 4) as year,
        SUBSTR(date, 6, 2) as month
    FROM raw_nikkei225
    WHERE Close IS NOT NULL
    ORDER BY date
""")
conn.commit()

stg = pd.read_sql("SELECT * FROM stg_nikkei225 ORDER BY date DESC LIMIT 5", conn)
print("  【stg_nikkei225（最新5行）】")
print(f"  {stg.to_string(index=False)}")

stg_count = pd.read_sql("SELECT COUNT(*) as cnt FROM stg_nikkei225", conn)
print(f"\n  ✓ stg_nikkei225 作成完了: {stg_count['cnt'][0]} 行")

[STAGING]生データの整理
  【stg_nikkei225（最新5行）】
        date  open_price  high_price  low_price  close_price    volume year month
2026-04-02    54066.83    54258.48   52415.11     52469.82         0 2026    04
2026-04-01    51959.47    53739.68   51902.84     53739.68 165900000 2026    04
2026-03-31    51382.53    52169.01   50558.91     51063.72 174500000 2026    03
2026-03-30    52054.68    52054.68   50566.99     51885.85 183100000 2026    03
2026-03-27    53239.59    53714.90   52516.92     53373.07 170500000 2026    03

  ✓ stg_nikkei225 作成完了: 1766 行


## 2. Intermediate 層:ビジネスロジックの適用

##　役割
- staging のデータにビジネスロジックを適用
- 移動平均、日次リターン、シグナルなどを計算
- ここが「分析の知恵」を入れる層

In [3]:
# Intermediate 層 1:日次リターン　+ 移動平均
print("[INTERMEDIATE] ビジネスロジック適用...")

conn.execute("DROP TABLE IF EXISTS int_daily_metrics")

conn.execute("""
    CREATE TABLE int_daily_metrics AS
    SELECT 
        date,
        close_price,
        volume,
        year,
        month,
        ROUND(
            (close_price - LAG(close_price) OVER (ORDER BY date))
            / LAG(close_price) OVER (ORDER BY date) * 100,
            4
        ) as daily_return,
        ROUND(AVG(close_price) OVER (
            ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
        ), 2) as ma_20,
        ROUND(AVG(close_price) OVER (
            ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW
        ), 2) as ma_200,
        ROUND(AVG(volume) OVER (
            ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
        ), 0) as avg_volume_20
    FROM stg_nikkei225
    ORDER BY date
""")
conn.commit()

int_sample = pd.read_sql("""
    SELECT date, close_price, daily_return, ma_20, ma_200
    FROM int_daily_metrics 
    ORDER BY date DESC LIMIT 5
""", conn)
print("  【int_daily_metrics（最新5行）】")
print(f"  {int_sample.to_string(index=False)}")

int_count = pd.read_sql("SELECT COUNT(*) as cnt FROM int_daily_metrics", conn)
print(f"\n  ✓ int_daily_metrics 作成完了: {int_count['cnt'][0]} 行")

[INTERMEDIATE] ビジネスロジック適用...
  【int_daily_metrics（最新5行）】
        date  close_price  daily_return    ma_20   ma_200
2026-04-02     52469.82       -2.3630 53544.53 47857.35
2026-04-01     53739.68        5.2404 53633.32 47783.71
2026-03-31     51063.72       -1.5845 53760.28 47702.78
2026-03-30     51885.85       -2.7865 54109.96 47636.20
2026-03-27     53373.07       -0.4302 54458.18 47564.01

  ✓ int_daily_metrics 作成完了: 1766 行


In [4]:
# Intermediate層2: トレーディングシグナル
conn.execute("DROP TABLE IF EXISTS int_trading_signals")

conn.execute("""
    CREATE TABLE int_trading_signals AS
    SELECT 
        date,
        close_price,
        ma_20,
        ma_200,
        daily_return,
        CASE 
            WHEN ma_20 > ma_200 
                 AND LAG(ma_20) OVER (ORDER BY date) <= LAG(ma_200) OVER (ORDER BY date)
            THEN 'BUY'
            WHEN ma_20 < ma_200 
                 AND LAG(ma_20) OVER (ORDER BY date) >= LAG(ma_200) OVER (ORDER BY date)
            THEN 'SELL'
            ELSE NULL
        END as signal,
        CASE 
            WHEN ma_20 > ma_200 THEN 'BULLISH'
            ELSE 'BEARISH'
        END as trend
    FROM int_daily_metrics
    WHERE ma_200 IS NOT NULL
    ORDER BY date
""")
conn.commit()

signals = pd.read_sql("""
    SELECT * FROM int_trading_signals 
    WHERE signal IS NOT NULL
    ORDER BY date
""", conn)
print("  【int_trading_signals（シグナルのみ）】")
print(f"  {signals.to_string(index=False)}")
print(f"\n  BUY:  {(signals['signal'] == 'BUY').sum()} 回")
print(f"  SELL: {(signals['signal'] == 'SELL').sum()} 回")

  【int_trading_signals（シグナルのみ）】
        date  close_price    ma_20   ma_200  daily_return signal   trend
2019-02-04     20883.77 20543.00 20496.28        0.4588    BUY BULLISH
2019-06-03     20410.88 21178.00 21237.28       -0.9238   SELL BEARISH
2019-07-03     21638.16 21236.55 21226.41       -0.5337    BUY BULLISH
2019-08-13     20455.44 21235.56 21246.88       -1.1089   SELL BEARISH
2019-09-24     22098.84 21239.12 21215.72        0.0895    BUY BULLISH
2020-03-11     19416.06 22053.53 22158.02       -2.2704   SELL BEARISH
2020-06-12     22305.48 21803.14 21749.44       -0.7450    BUY BULLISH
2021-08-06     27820.04 27891.58 27927.46        0.3315   SELL BEARISH
2021-09-13     30447.37 28511.77 28455.04        0.2157    BUY BULLISH
2021-10-22     28804.85 28760.68 28803.61        0.3353   SELL BEARISH
2021-11-05     29611.57 28931.95 28875.39       -0.6135    BUY BULLISH
2021-12-13     28640.49 28840.85 28877.67        0.7129   SELL BEARISH
2022-08-10     27819.33 27623.21 27550.45  

## 4. Marts 層:最終分析テーブル

###役割
-分析者やBIツールが直接使う最終テーブル
- 「このテーブルをSELECT するだけで分析が完結する」状態にする
- 集計済み・計算済みのデータを提供する

In [5]:
# Marts層 1:月次サマリー
print("[MARTS] 最終分析テーブル作成...")

conn.execute("DROP TABLE IF EXISTS mart_monthly_summary")

conn.execute("""
    CREATE TABLE mart_monthly_summary AS
    SELECT 
        year,
        month,
        year || '-' || month as year_month,
        COUNT(*) as trading_days,
        ROUND(AVG(close_price), 2) as avg_close,
        ROUND(MAX(close_price), 2) as max_close,
        ROUND(MIN(close_price), 2) as min_close,
        ROUND((MAX(close_price) - MIN(close_price)) / MIN(close_price) * 100, 2) as price_range_pct,
        ROUND(AVG(daily_return), 4) as avg_daily_return,
        ROUND(AVG(volume), 0) as avg_volume,
        SUM(CASE WHEN daily_return > 0 THEN 1 ELSE 0 END) as up_days,
        SUM(CASE WHEN daily_return < 0 THEN 1 ELSE 0 END) as down_days,
        ROUND(
            CAST(SUM(CASE WHEN daily_return > 0 THEN 1 ELSE 0 END) AS REAL) 
            / COUNT(*) * 100, 
            1
        ) as up_day_pct
    FROM int_daily_metrics
    WHERE daily_return IS NOT NULL
    GROUP BY year, month
    ORDER BY year, month
""")
conn.commit()

mart_monthly = pd.read_sql("""
    SELECT year_month, trading_days, avg_close, price_range_pct, avg_daily_return, up_day_pct
    FROM mart_monthly_summary 
    ORDER BY year_month DESC LIMIT 12
""", conn)
print("  【mart_monthly_summary（直近12ヶ月）】")
print(f"  {mart_monthly.to_string(index=False)}")

[MARTS] 最終分析テーブル作成...
  【mart_monthly_summary（直近12ヶ月）】
  year_month  trading_days  avg_close  price_range_pct  avg_daily_return  up_day_pct
   2026-04             2   53104.75             2.42            1.4387        50.0
   2026-03            21   53964.90            13.70           -0.6467        33.3
   2026-02            18   56480.85            11.77            0.5614        55.6
   2026-01            19   53077.27             6.31            0.3129        52.6
   2025-12            22   50162.42             4.14            0.0133        54.5
   2025-11            18   50111.11             6.10           -0.2207        50.0
   2025-10            22   48521.07            17.64            0.7168        63.6
   2025-09            20   44218.54             9.10            0.2565        60.0
   2025-08            20   42299.85             8.50            0.2026        60.0
   2025-07            22   40173.04             6.00            0.0698        40.9
   2025-06            21   384

In [6]:
# Marts 層2:　トレード実績サマリー
conn.execute("DROP TABLE IF EXISTS mart_trade_performance")

# シグナルからトレードペアを Python で作成して SQL に保存
signals_df = pd.read_sql("""
    SELECT date, close_price, signal 
    FROM int_trading_signals 
    WHERE signal IS NOT NULL
    ORDER BY date
""", conn)

buy_rows = signals_df[signals_df['signal'] == 'BUY'].reset_index(drop=True)
sell_rows = signals_df[signals_df['signal'] == 'SELL'].reset_index(drop=True)

trades = []
for i, buy in buy_rows.iterrows():
    future_sells = sell_rows[sell_rows['date'] > buy['date']]
    if len(future_sells) > 0:
        sell = future_sells.iloc[0]
        trades.append({
            'trade_id': i + 1,
            'buy_date': buy['date'],
            'sell_date': sell['date'],
            'buy_price': buy['close_price'],
            'sell_price': sell['close_price'],
            'return_pct': round((sell['close_price'] - buy['close_price']) / buy['close_price'] * 100, 4),
            'holding_days': (pd.Timestamp(sell['date']) - pd.Timestamp(buy['date'])).days
        })

trades_df = pd.DataFrame(trades)
trades_df.to_sql('mart_trade_performance', conn, if_exists='replace', index=False)

print("\n  【mart_trade_performance（全トレード）】")
print(f"  {trades_df.to_string(index=False)}")

print(f"\n  トレード数: {len(trades_df)}")
print(f"  勝率:       {(trades_df['return_pct'] > 0).mean() * 100:.1f}%")
print(f"  平均リターン: {trades_df['return_pct'].mean():.2f}%")
print(f"  平均保有日数: {trades_df['holding_days'].mean():.0f} 日")


  【mart_trade_performance（全トレード）】
   trade_id   buy_date  sell_date  buy_price  sell_price  return_pct  holding_days
        1 2019-02-04 2019-06-03   20883.77    20410.88     -2.2644           119
        2 2019-07-03 2019-08-13   21638.16    20455.44     -5.4659            41
        3 2019-09-24 2020-03-11   22098.84    19416.06    -12.1399           169
        4 2020-06-12 2021-08-06   22305.48    27820.04     24.7229           420
        5 2021-09-13 2021-10-22   30447.37    28804.85     -5.3946            39
        6 2021-11-05 2021-12-13   29611.57    28640.49     -3.2794            38
        7 2022-08-10 2022-10-04   27819.33    26992.21     -2.9732            55
        8 2022-11-09 2022-12-29   27716.43    26093.67     -5.8549            50
        9 2023-02-14 2024-08-19   27602.77    37388.62     35.4524           552
       10 2024-09-05 2024-09-13   36657.09    36581.76     -0.2055             8
       11 2024-10-11 2024-12-06   39605.80    39091.17     -1.2994      

## 4. データ品質テスト (dbt testの再現)

### dbtのテスト機能
dbt　では以下のテストを自動実行できる:
- not_null:NULLがないか
- unique:重複がないか
- accepted_values:想定外の値がないか
- relationships:外部キーの整合性

### Pythonで同じことを実装する

In [7]:
# ===== データ品質テスト =====
print("=" * 60)
print("[TEST] データ品質テスト実行")
print("=" * 60)

test_results = []

def run_test(test_name, query, expected_value=0):
    """テストを実行して結果を記録"""
    result = pd.read_sql(query, conn)
    actual_value = result.iloc[0, 0]
    passed = actual_value == expected_value
    status = "✓ PASS" if passed else "✗ FAIL"
    test_results.append({
        'test': test_name,
        'expected': expected_value,
        'actual': actual_value,
        'status': status
    })
    print(f"  {status}: {test_name} (expected={expected_value}, actual={actual_value})")
    return passed

# ----- stg_nikkei225 のテスト -----
print("\n[stg_nikkei225]")

run_test(
    "stg: close_price に NULL がない",
    "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price IS NULL"
)

run_test(
    "stg: date が重複していない",
    "SELECT COUNT(*) FROM (SELECT date, COUNT(*) as c FROM stg_nikkei225 GROUP BY date HAVING c > 1)"
)

run_test(
    "stg: close_price が正の値",
    "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price <= 0"
)

# ----- int_daily_metrics のテスト -----
print("\n[int_daily_metrics]")

run_test(
    "int: date が重複していない",
    "SELECT COUNT(*) FROM (SELECT date, COUNT(*) as c FROM int_daily_metrics GROUP BY date HAVING c > 1)"
)

run_test(
    "int: ma_20 が close_price の妥当な範囲内",
    """SELECT COUNT(*) FROM int_daily_metrics 
       WHERE ma_20 IS NOT NULL 
       AND (ma_20 > close_price * 2 OR ma_20 < close_price * 0.5)"""
)

# ----- int_trading_signals のテスト -----
print("\n[int_trading_signals]")

run_test(
    "signals: signal は BUY/SELL/NULL のみ",
    """SELECT COUNT(*) FROM int_trading_signals 
       WHERE signal IS NOT NULL AND signal NOT IN ('BUY', 'SELL')"""
)

run_test(
    "signals: trend は BULLISH/BEARISH のみ",
    """SELECT COUNT(*) FROM int_trading_signals 
       WHERE trend NOT IN ('BULLISH', 'BEARISH')"""
)

# ----- mart_monthly_summary のテスト -----
print("\n[mart_monthly_summary]")

run_test(
    "mart: year_month が重複していない",
    "SELECT COUNT(*) FROM (SELECT year_month, COUNT(*) as c FROM mart_monthly_summary GROUP BY year_month HAVING c > 1)"
)

run_test(
    "mart: up_day_pct が 0〜100 の範囲",
    "SELECT COUNT(*) FROM mart_monthly_summary WHERE up_day_pct < 0 OR up_day_pct > 100"
)

# ----- 結果サマリー -----
print("\n" + "=" * 60)
results_df = pd.DataFrame(test_results)
passed = (results_df['status'] == '✓ PASS').sum()
total = len(results_df)
print(f"テスト結果: {passed}/{total} 通過")
if passed == total:
    print("✓ 全テスト通過！データ品質に問題なし")
else:
    print("✗ 一部テストが失敗しています。確認してください")
print("=" * 60)

[TEST] データ品質テスト実行

[stg_nikkei225]
  ✓ PASS: stg: close_price に NULL がない (expected=0, actual=0)
  ✓ PASS: stg: date が重複していない (expected=0, actual=0)
  ✓ PASS: stg: close_price が正の値 (expected=0, actual=0)

[int_daily_metrics]
  ✓ PASS: int: date が重複していない (expected=0, actual=0)
  ✓ PASS: int: ma_20 が close_price の妥当な範囲内 (expected=0, actual=0)

[int_trading_signals]
  ✓ PASS: signals: signal は BUY/SELL/NULL のみ (expected=0, actual=0)
  ✓ PASS: signals: trend は BULLISH/BEARISH のみ (expected=0, actual=0)

[mart_monthly_summary]
  ✓ PASS: mart: year_month が重複していない (expected=0, actual=0)
  ✓ PASS: mart: up_day_pct が 0〜100 の範囲 (expected=0, actual=0)

テスト結果: 9/9 通過
✓ 全テスト通過！データ品質に問題なし


## 5.パイプライン全体の可視化

In [8]:
# パイプライン全体のテーブル一覧
print("【dbt スタイル パイプライン全体像】\n")

layers = {
    'RAW（生データ）': ['raw_nikkei225'],
    'STAGING（整理）': ['stg_nikkei225'],
    'INTERMEDIATE（加工）': ['int_daily_metrics', 'int_trading_signals'],
    'MARTS（最終）': ['mart_monthly_summary', 'mart_trade_performance']
}

for layer_name, tables in layers.items():
    print(f"═══ {layer_name} ═══")
    for table in tables:
        count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM [{table}]", conn)
        cols = pd.read_sql(f"PRAGMA table_info([{table}])", conn)
        print(f"  {table}")
        print(f"    {count['cnt'][0]} 行 / {len(cols)} 列")
    print()

【dbt スタイル パイプライン全体像】

═══ RAW（生データ） ═══
  raw_nikkei225
    1766 行 / 6 列

═══ STAGING（整理） ═══
  stg_nikkei225
    1766 行 / 8 列

═══ INTERMEDIATE（加工） ═══
  int_daily_metrics
    1766 行 / 9 列
  int_trading_signals
    1766 行 / 7 列

═══ MARTS（最終） ═══
  mart_monthly_summary
    88 行 / 13 列
  mart_trade_performance
    12 行 / 7 列



# 6.　自分なりのまとめ

### dbtの層構造は、データを段階的に加工していく設計パターン。
- raw: yfinanceから取得した生データ。一切触らず保管する(元に戻せなくなるリスクを防ぐ)
- staging:列名の統一、型変換、欠損値処理など「整理」だけを行う層
- intermediate: 移動平均やリターン計算など「ビジネスロジック」を適用する層。分析の知恵を入れる場所
- marts: アナリストやサイエンティストが直接使う最終テーブル。SELECTするだけで分析が完結する状態

この層構造の利点は作業の標準化。一度パイプラインを作れば、
少しの作業で同レベルの分析と品質チェックが繰り返し実行できる。
コードを毎回書き直す必要がなくなる。

Data Martはデータウェアハウスから特定の分析目的に合わせて切り出した最終テーブルのこと。
今回のmart_monthly_summaryは月次分析用、mart_trade_performanceはトレード分析用のData Martにあたる。

データ品質テストは各層のテーブルに対して「NULLがないか」「重複がないか」
「値が妥当な範囲か」を自動チェックする仕組み。
dbtではYAMLファイルに設定を書くだけで自動実行されるが、今回はPython + SQLで同党のテストを9件実装し、全件通過を確認した。
Day15-16でdbtをインストールしたら、同じテストをdbtのシンプルな構文で置き換える予定

### 出力結果について
Staging 層：raw データから列名を統一（Close → close_price 等）し、1766行を整理。
生データは触らずそのまま保管している。

Intermediate層 ：2つのテーブルを作成。
int_daily_metrics（日次リターン + 移動平均）と int_trading_signals（売買シグナル + トレンド方向）。
BUY 13回、SELL 12回を検出。

Marts 層：月次サマリーとトレード実績の2つの Data Mart を作成。
月次サマリーでは 2026年3月の price_range_pct が 13.70% と高く、
ボラティリティの大きさが確認できる。
トレード実績は勝率16.7%、平均リターン+1.34%、平均保有136日。

データ品質テスト：9件全て通過。
NULL チェック、重複チェック、値の範囲チェック、accepted values チェックを実施し、
パイプライン全体のデータ品質に問題がないことを確認した。

パイプライン全体：raw（6列）→ staging（8列）→ intermediate（9列・7列）→ marts（13列・7列）
と、段階を経るごとにデータが加工・集約されていく流れを確認できた。


In [9]:
conn.close()
print("✓ データベース接続を閉じました")
print("✓ Notebook 7 完了")

✓ データベース接続を閉じました
✓ Notebook 7 完了
